# 01 — Coleta de dados recentes

Este notebook é o primeiro passo do fluxo **"prever a qualidade do ar 1 hora à frente, a partir de agora"**.

## O que será feito

1. Ler localização, variáveis e endpoints de `config/params.yaml`.
2. Consultar as APIs da Open-Meteo (qualidade do ar e meteorologia) para os últimos dias.
3. Descartar horas futuras (as APIs devolvem o restante do dia como previsão).
4. Inspecionar a cobertura e salvar uma cópia em `data/recent/`.

> A execução faz chamadas externas. A cópia salva é **sobrescrita a cada coleta**, pois representa sempre o momento atual. Os dados de qualidade do ar são modelados (CAMS), não medições de sensores.

Execute as células em ordem, usando o ambiente `.venv` do projeto. Depois siga para [`02_Features_Recentes.ipynb`](02_Features_Recentes.ipynb).

## 1. Preparar o ambiente e carregar a configuração

In [1]:
import os
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd()
if not (ROOT / 'config' / 'params.yaml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'params.yaml').is_file():
    raise FileNotFoundError('Não encontrei config/params.yaml. Abra o notebook a partir da raiz do projeto.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.aplicacao.Prever_Proxima_Hora import _coletar_dados_recentes

with (ROOT / 'config' / 'params.yaml').open('r', encoding='utf-8') as arquivo:
    config = yaml.safe_load(arquivo)

location = config['location']
aplicacao = config['application']
print(f"Ponto: {location['latitude']}, {location['longitude']} ({location['timezone']})")
print(f"Histórico solicitado: {aplicacao['recent_history_days']} dias")

Ponto: -23.514561, -46.186832 (America/Sao_Paulo)
Histórico solicitado: 3 dias


## 2. Consultar as APIs

A função junta qualidade do ar e meteorologia pelo horário (`time`) e mantém apenas horas até o momento atual.

In [2]:
dados = _coletar_dados_recentes(config)
print(f'Linhas coletadas: {len(dados)}')
print(f"Primeira hora: {dados['time'].iloc[0]}")
print(f"Última hora disponível (\"agora\"): {dados['time'].iloc[-1]}")

Linhas coletadas: 96
Primeira hora: 2026-10-17 00:00:00
Última hora disponível ("agora"): 2026-10-20 23:00:00


## 3. Inspecionar os dados

Confira se não há lacunas e se a última hora está preenchida. Valores ausentes na borda final indicam atraso da fonte.

In [3]:
intervalos = dados['time'].diff().dropna()
print('Horários consecutivos (1h):', bool((intervalos == intervalos.iloc[0]).all()))
nulos = dados.isna().sum()
print('Valores ausentes por coluna:')
print(nulos[nulos > 0] if nulos.sum() else 'nenhum')
dados.tail(6)

Horários consecutivos (1h): True
Valores ausentes por coluna:
nenhum


,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,pressure_msl
90,2026-10-20 18:00:00,10.5,7.4,108.0,4.0,1.1,55.0,15.7,82,0.0,14.8,1020.1
91,2026-10-20 19:00:00,9.5,7.0,111.0,4.8,1.0,47.0,14.6,88,0.0,10.9,1020.6
92,2026-10-20 20:00:00,8.8,6.7,113.0,5.4,1.0,40.0,14.0,92,0.0,7.0,1021.2
93,2026-10-20 21:00:00,8.6,6.5,113.0,5.7,0.9,35.0,13.7,95,0.0,6.9,1021.7
94,2026-10-20 22:00:00,8.8,6.7,107.0,5.3,0.8,35.0,13.6,95,0.0,8.0,1021.8
95,2026-10-20 23:00:00,8.7,6.6,99.0,4.4,0.7,37.0,13.3,96,0.0,7.9,1021.6


## 4. Salvar a cópia dos dados recentes

In [4]:
caminho_copia = ROOT / aplicacao['recent_snapshot']
caminho_copia.parent.mkdir(parents=True, exist_ok=True)
dados.to_csv(caminho_copia, index=False, encoding='utf-8')
print(f'Dados recentes salvos em: {caminho_copia}')

Dados recentes salvos em: c:\Projetos\Ciencia-de-Dados-e-Aprendizagem-de-Maquina\data\recent\dados_recentes.csv


## Próximo passo

Abra [`02_Features_Recentes.ipynb`](02_Features_Recentes.ipynb) para calcular o IQAr e as features usadas pelo modelo. A alternativa sem notebook é `python -m src.aplicacao.Prever_Proxima_Hora`.